# Experiment 3: Dynamics Comparison

**Goal**: Compare the F0 dynamics coefficients ($\beta_3$ = velocity,
$\beta_4$ = acceleration) across all methods and flag which are
statistically significant.

**Hypothesis**: WORLD's $\beta_3$ and $\beta_4$ should be near zero or
nonsignificant (WORLD does not model dynamics). DDSP should be closer
to the reference violin's values.

**Inputs**:
- Reference models from Experiment 1 (`exp1_ref_coupling_models.pkl`)
- Method models from Experiment 2 (`exp2_method_models.pkl`)

**Outputs**: `artifacts/evaluation/exp3_*.csv` and figures

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
EVAL_DIR = PROJECT_ROOT / "artifacts" / "evaluation"
FIG_DIR  = EVAL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SIGNIFICANCE_LEVEL = 0.05

print(f"Eval dir: {EVAL_DIR}")

## 1. Load models

In [ ]:
with open(EVAL_DIR / "exp1_ref_coupling_models.pkl", "rb") as fh:
    ref_bundle = pickle.load(fh)

with open(EVAL_DIR / "exp2_method_models.pkl", "rb") as fh:
    method_models = pickle.load(fh)

ref_models  = ref_bundle["models"]
DESCRIPTORS = ref_bundle["descriptors"]

# Combine: reference + methods
all_models = {"reference": ref_models, **method_models}

print(f"Methods: {list(all_models.keys())}")
print(f"Descriptors: {DESCRIPTORS}")

## 2. Extract dynamics coefficients ($\beta_3$, $\beta_4$) and their p-values

In [ ]:
rows = []
for method_name, models in all_models.items():
    for desc in DESCRIPTORS:
        m = models[desc]
        coeffs_b = m["coeffs_b"]
        pvals_b  = m["coeff_pvalues_b"]
        rows.append({
            "method":     method_name,
            "descriptor": desc,
            "b3_velocity":     coeffs_b[3],
            "b4_acceleration": coeffs_b[4],
            "b3_pvalue":       pvals_b[3],
            "b4_pvalue":       pvals_b[4],
            "b3_significant":  pvals_b[3] < SIGNIFICANCE_LEVEL,
            "b4_significant":  pvals_b[4] < SIGNIFICANCE_LEVEL,
        })

df_dyn = pd.DataFrame(rows)
print(f"Dynamics table: {df_dyn.shape}")
display(df_dyn.round(6))

## 3. Pivot tables: coefficients side by side

In [ ]:
print("=== \u03b23 (F0 velocity) ===")
pivot_b3 = df_dyn.pivot(index="descriptor", columns="method", values="b3_velocity")
display(pivot_b3.round(6))

print("\n=== \u03b24 (F0 acceleration) ===")
pivot_b4 = df_dyn.pivot(index="descriptor", columns="method", values="b4_acceleration")
display(pivot_b4.round(6))

print("\n=== Significance flags (p < 0.05) ===")
sig_b3 = df_dyn.pivot(index="descriptor", columns="method", values="b3_significant")
sig_b4 = df_dyn.pivot(index="descriptor", columns="method", values="b4_significant")
# Combine into a single display
sig_display = sig_b3.astype(str).add(" / ").add(sig_b4.astype(str))
sig_display.columns = [f"{c} (b3/b4)" for c in sig_display.columns]
display(sig_display)

## 4. Plots

In [ ]:
# Grouped bar chart for b3 (velocity)
methods = list(all_models.keys())
n_methods = len(methods)
x = np.arange(len(DESCRIPTORS))
width = 0.8 / n_methods

colors = {"reference": "0.6", "ddsp": "steelblue", "baseline": "tomato"}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# b3 (velocity)
ax = axes[0]
for j, method in enumerate(methods):
    vals = pivot_b3[method].values
    sigs = sig_b3[method].values
    color = colors.get(method, f"C{j}")
    bars = ax.bar(x + j * width - (n_methods - 1) * width / 2, vals, width,
                  label=method, color=color, edgecolor="black", linewidth=0.3)
    # Mark significant bars
    for bar, sig in zip(bars, sigs):
        if sig:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    "*", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(DESCRIPTORS, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("\u03b23 (velocity coefficient)")
ax.set_title("\u03b23 (F0 velocity) by method")
ax.axhline(0, color="black", lw=0.5, ls="--")
ax.legend(fontsize=8)

# b4 (acceleration)
ax = axes[1]
for j, method in enumerate(methods):
    vals = pivot_b4[method].values
    sigs = sig_b4[method].values
    color = colors.get(method, f"C{j}")
    bars = ax.bar(x + j * width - (n_methods - 1) * width / 2, vals, width,
                  label=method, color=color, edgecolor="black", linewidth=0.3)
    for bar, sig in zip(bars, sigs):
        if sig:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    "*", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(DESCRIPTORS, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("\u03b24 (acceleration coefficient)")
ax.set_title("\u03b24 (F0 acceleration) by method")
ax.axhline(0, color="black", lw=0.5, ls="--")
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / "exp3_dynamics_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"* = significant at p < {SIGNIFICANCE_LEVEL}")

## 5. Dynamics-only coefficient distance

How far are each method's $(\beta_3, \beta_4)$ from the reference?

In [ ]:
dyn_dist_rows = []
for method in methods:
    if method == "reference":
        continue
    for desc in DESCRIPTORS:
        b_ref = ref_models[desc]["coeffs_b"][3:5]
        b_out = all_models[method][desc]["coeffs_b"][3:5]
        dyn_dist_rows.append({
            "method": method,
            "descriptor": desc,
            "dynamics_distance": float(np.linalg.norm(b_out - b_ref)),
        })

df_dyn_dist = pd.DataFrame(dyn_dist_rows)
pivot_dd = df_dyn_dist.pivot(index="descriptor", columns="method", values="dynamics_distance")

print("Dynamics coefficient distance ||[b3,b4]_output - [b3,b4]_ref||:")
display(pivot_dd.round(6))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
pivot_dd.plot.bar(ax=ax, color=[colors.get(c, "C0") for c in pivot_dd.columns],
                  edgecolor="black", linewidth=0.3)
ax.set_ylabel("L2 distance")
ax.set_title("Dynamics Coefficient Distance to Reference")
ax.set_xticklabels(DESCRIPTORS, rotation=45, ha="right", fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "exp3_dynamics_distance.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary

In [ ]:
# Count significant dynamics coefficients per method
sig_counts = df_dyn.groupby("method")[["b3_significant", "b4_significant"]].sum()
sig_counts.columns = ["n_significant_b3", "n_significant_b4"]
sig_counts["total_descriptors"] = len(DESCRIPTORS)
print("Significant dynamics coefficients per method:")
display(sig_counts)

# Mean dynamics distance
if not df_dyn_dist.empty:
    mean_dd = df_dyn_dist.groupby("method")["dynamics_distance"].mean()
    print("\nMean dynamics distance to reference:")
    display(mean_dd.round(6))

## 7. Save results

In [ ]:
df_dyn.to_csv(EVAL_DIR / "exp3_dynamics_coefficients.csv", index=False)
if not df_dyn_dist.empty:
    df_dyn_dist.to_csv(EVAL_DIR / "exp3_dynamics_distance.csv", index=False)

print("Saved:")
for f in ["exp3_dynamics_coefficients.csv", "exp3_dynamics_distance.csv"]:
    p = EVAL_DIR / f
    if p.exists():
        print(f"  {p}")